# Customer Discovery & Stakeholder Communication — Applied

**FDE Delivery · Week 23b**

Offline notebook for the insurance-underwriter AI assistant engagement: stakeholder map, discovery brief, 14-feature scope proposal, executive summary, weekly steering packs, and go-live engagement wrap.

## 1. Engagement timeline and stakeholder map

```mermaid
quadrantChart
  title Insurance underwriter assistant stakeholders
  x-axis Low influence --> High influence
  y-axis Low interest --> High interest
  quadrant-1 Manage closely
  quadrant-2 Keep informed
  quadrant-3 Monitor
  quadrant-4 Keep satisfied
  CUO: [0.92, 0.94]
  Commercial SME: [0.66, 0.90]
  25 Underwriters: [0.34, 0.82]
  CISO: [0.86, 0.54]
  DPO: [0.88, 0.58]
  Platform Lead: [0.78, 0.62]
  Procurement: [0.74, 0.36]
```

Discovery created the owner map that later made architecture, SRB, LLMOps, and handoff possible.

In [ ]:
from __future__ import annotations
from enum import Enum
from typing import Literal
from pydantic import BaseModel, ConfigDict, Field
print('Week 23b notebook ready: insurance-underwriter discovery and stakeholder communication.')

In [ ]:
class Influence(str, Enum): low='low'; high='high'
class Interest(str, Enum): low='low'; high='high'
class Stakeholder(BaseModel):
    model_config = ConfigDict(extra='forbid')
    name: str; role: str; interest: Interest; influence: Influence; concern: str
    @property
    def quadrant(self):
        if self.interest == Interest.high and self.influence == Influence.high: return 'Manage Closely'
        if self.interest == Interest.low and self.influence == Influence.high: return 'Keep Satisfied'
        if self.interest == Interest.high and self.influence == Influence.low: return 'Keep Informed'
        return 'Monitor'
class SuccessCriterion(BaseModel):
    statement: str; metric: str; baseline: str; target: str; timeframe: str; measurement: str
class Constraint(BaseModel):
    category: str; statement: str
class Risk(BaseModel):
    risk: str; likelihood: str; impact: str; mitigation: str; owner: str
class DiscoveryBrief(BaseModel):
    customer: str; vague_ask: str; business_problem: str; jtbd: str; current_pain: str
    stakeholders: list[Stakeholder]; success_criteria: list[SuccessCriterion]; constraints: list[Constraint]; risks: list[Risk]
    def render(self):
        lines=[f'# DiscoveryBrief — {self.customer}', f'Ask: {self.vague_ask}', f'Problem: {self.business_problem}', f'JTBD: {self.jtbd}', f'Pain: {self.current_pain}', '', '## Success criteria']
        lines += [f'- {c.statement}: {c.baseline} -> {c.target} ({c.timeframe}); measured by {c.measurement}' for c in self.success_criteria]
        lines += ['', '## Stakeholders'] + [f'- {s.name} ({s.role}) — {s.quadrant}; {s.concern}' for s in self.stakeholders]
        lines += ['', '## Constraints'] + [f'- {c.category}: {c.statement}' for c in self.constraints]
        lines += ['', '## Risks'] + [f'- {r.risk} [{r.likelihood}/{r.impact}] owner={r.owner}; mitigation={r.mitigation}' for r in self.risks]
        return '\n'.join(lines)

In [ ]:
brief = DiscoveryBrief(
 customer='Mid-sized insurance carrier — commercial underwriting', vague_ask='We want an AI assistant for underwriters.',
 business_problem='Reduce median time-to-quote for complex commercial policies from 4 hours to 90 minutes within 6 months of go-live.',
 jtbd='When I am underwriting a complex commercial policy, I want trusted policy, precedent, and regulatory evidence with citations, so I can quote faster while preserving human judgment and auditability.',
 current_pain='Underwriters spend roughly 60% of the day searching 15 years of memos, 40k policy docs, Guidewire context, and 3 regulatory feeds.',
 stakeholders=[Stakeholder(name='Chief Underwriting Officer', role='exec sponsor', interest='high', influence='high', concern='business case and adoption'), Stakeholder(name='Senior Underwriter — Commercial', role='SME', interest='high', influence='high', concern='precedent quality'), Stakeholder(name='25 underwriters', role='end users', interest='high', influence='low', concern='trust and workflow fit'), Stakeholder(name='DPO', role='privacy', interest='low', influence='high', concern='no third-party leakage'), Stakeholder(name='Azure Platform Lead', role='platform', interest='high', influence='high', concern='operable handoff')],
 success_criteria=[SuccessCriterion(statement='Reduce median complex quote time', metric='workflow time', baseline='4 hours', target='90 minutes', timeframe='within 6 months', measurement='workflow instrumentation'), SuccessCriterion(statement='Maintain groundedness', metric='golden-set score', baseline='none', target='>= 92%', timeframe='before and after launch', measurement='SME-owned 200-question eval'), SuccessCriterion(statement='Control run cost', metric='monthly baseline run cost', baseline='new', target='<=$3,300/month', timeframe='50-user launch', measurement='FinOps dashboard')],
 constraints=[Constraint(category='technical', statement='Azure-only private endpoint architecture'), Constraint(category='regulatory', statement='approved-region data residency and no third-party leakage'), Constraint(category='integration', statement='Guidewire PolicyCenter context'), Constraint(category='timeline/budget', statement='6 months and $850k ceiling')],
 risks=[Risk(risk='Hallucination creates bad recommendation', likelihood='medium', impact='high', mitigation='decision support, citations, review queue', owner='CUO/FDE'), Risk(risk='Provider outage halts workflow', likelihood='medium', impact='medium', mitigation='fallback and runbook', owner='Platform'), Risk(risk='Adoption stalls from distrust', likelihood='medium', impact='high', mitigation='SME golden set and working sessions', owner='Commercial SME')])
print(brief.render())

In [ ]:
class Feature(BaseModel):
    id: str; name: str; category: str; business_value: int; ai_confidence: float; reach: int; effort_days: int; must: bool=False
    @property
    def score(self): return round(self.reach*self.business_value*self.ai_confidence/self.effort_days, 2)
features=[Feature(id='policy_qa',name='Policy history Q&A with citations',category='core',business_value=10,ai_confidence=.86,reach=25,effort_days=28,must=True), Feature(id='memo_search',name='15-year memo precedent search',category='core',business_value=9,ai_confidence=.82,reach=25,effort_days=24,must=True), Feature(id='reg_feeds',name='Regulatory cross-reference',category='core',business_value=8,ai_confidence=.78,reach=20,effort_days=22), Feature(id='guidewire',name='Guidewire context filters',category='integration',business_value=9,ai_confidence=.80,reach=25,effort_days=20,must=True), Feature(id='citation_export',name='Cited audit-note export',category='workflow',business_value=9,ai_confidence=.90,reach=25,effort_days=16), Feature(id='audit',name='Immutable audit log',category='governance',business_value=8,ai_confidence=.88,reach=25,effort_days=18,must=True), Feature(id='feedback',name='Underwriter feedback capture',category='eval',business_value=7,ai_confidence=.90,reach=25,effort_days=12), Feature(id='evals',name='Golden-set eval framework',category='eval',business_value=9,ai_confidence=.92,reach=25,effort_days=18,must=True), Feature(id='review_queue',name='Refusal and senior-review queue',category='risk',business_value=8,ai_confidence=.84,reach=20,effort_days=16,must=True), Feature(id='training',name='Training analytics',category='adoption',business_value=5,ai_confidence=.76,reach=12,effort_days=14), Feature(id='cache',name='Semantic cache',category='ops',business_value=6,ai_confidence=.82,reach=25,effort_days=14), Feature(id='auto_decisions',name='Automatic underwriting decisions',category='autonomy',business_value=10,ai_confidence=.35,reach=8,effort_days=40), Feature(id='broker_email',name='Broker email drafting',category='workflow',business_value=5,ai_confidence=.55,reach=10,effort_days=24), Feature(id='portfolio',name='Portfolio analytics',category='analytics',business_value=6,ai_confidence=.50,reach=8,effort_days=30)]
for f in sorted(features, key=lambda x: x.score, reverse=True):
    print(f'{f.name:38s} score={f.score:5.2f} must={f.must}')
print('\nOption A Complete Vision: $1.2M, 9 months, high autonomy risk')
print('Option B Balanced MVP: $780k, 6 months, 8 features + eval framework — APPROVE')
print('Option C Foundation Only: $480k, 4 months, retrieval/citations only')

In [ ]:
exec_summary = """BLUF: An AI-powered underwriting assistant will reduce median time-to-quote for complex commercial policies from 4h to 90min within 6 months, at a total delivery cost of $780k and monthly run cost of $3,300 at baseline scale, with measurable groundedness and audit controls that satisfy the DPO and Model Risk Committee.

Situation: complex commercial underwriters spend about 60% of their day searching policy docs, 15 years of memos, and 3 regulatory feeds.
Complication: unmanaged AI would create hallucination, privacy, and model-risk exposure.
Question: can we reduce quote cycle time without increasing underwriting or privacy risk?
Answer: approve Option B Balanced MVP; reject Option A as too broad/risky and Option C as too narrow for the business case."""
print(exec_summary)
print('\nDemo arc: setup=real Tuesday search pain; confrontation=45-second cited answer; resolution=4h→90m target, groundedness >=92%, audit log shown live.')

In [ ]:
Status = Literal['green','yellow','red']
class Sprint(BaseModel): week:int; goals:list[str]; delivered:list[str]; blockers:list[str]; done:int; total:int
class RiskItem(BaseModel): risk:str; likelihood:str; impact:str; mitigation:str; owner:str; status:Status; review_date:str
class Decision(BaseModel): date:str; what:str; who:str; rationale:str
class Comms(BaseModel): stakeholder:str; cadence:str; last_update:str; status:Status; note:str
class Dashboard(BaseModel):
    title:str; sprint:Sprint; risks:list[RiskItem]; decisions:list[Decision]; comms:list[Comms]; demo:str; exec_message:str
    def render(self):
        pct=self.sprint.done/self.sprint.total*100
        lines=[f'# Week {self.sprint.week} steering pack — {self.title}', f'Burn-up: {self.sprint.done}/{self.sprint.total} ({pct:.0f}%). {self.exec_message}', '## This week / next week']
        lines += ['Goals:']+[f'- {x}' for x in self.sprint.goals]+['Delivered:']+[f'- {x}' for x in self.sprint.delivered]+['Blockers:']+([f'- {x}' for x in self.sprint.blockers] or ['- none'])
        lines += ['## Top risks']+[f'- [{r.status.upper()}] {r.risk} — {r.mitigation} ({r.owner})' for r in self.risks[:3]]
        lines += ['## Decisions']+[f'- {d.date}: {d.what} — {d.rationale}' for d in self.decisions]
        lines += ['## Demo', self.demo, '## Cadence']+[f'- [{c.status.upper()}] {c.stakeholder}: {c.note}' for c in self.comms]
        return '\n'.join(lines)
def week8():
    return Dashboard(title='Insurance Underwriter AI Assistant', sprint=Sprint(week=8, goals=['Tune refusal threshold','Finish Guidewire filters'], delivered=['Legitimate refusal reproduced','PTU forecast reviewed'], blockers=['Production quota confirmation'], done=31, total=92), risks=[RiskItem(risk='Refusal threshold blocks legitimate questions', likelihood='medium', impact='medium', mitigation='Tune with evals and add case to golden set', owner='FDE/eval', status='yellow', review_date='week9'), RiskItem(risk='PTU estimate 30% high', likelihood='medium', impact='medium', mitigation='Stay Standard until 200-user trigger', owner='FDE/platform', status='yellow', review_date='week10'), RiskItem(risk='Hallucination', likelihood='medium', impact='high', mitigation='Citations, refusal, review queue', owner='SME', status='yellow', review_date='weekly')], decisions=[Decision(date='week8', what='Lower refusal threshold', who='FDE+SME', rationale='Valid-answer rate improves; hallucination remains within budget'), Decision(date='week8', what='Stay Standard Azure OpenAI', who='FDE+CFO', rationale='PTU not economical at 50 users')], comms=[Comms(stakeholder='CUO', cadence='weekly', last_update='today', status='green', note='same-day refusal tradeoff note'), Comms(stakeholder='CFO', cadence='as needed', last_update='today', status='green', note='PTU memo sent')], demo='52-second cited answer plus refusal-threshold eval numbers.', exec_message='Measured threshold and cost decisions; no timeline change.')
print(week8().render())

In [ ]:
d = week8().model_copy(deep=True)
d.sprint = Sprint(week=19, goals=['Close SRB red-team evidence','Complete go-live checklist','Move training analytics to v2'], delivered=['SRB feedback triaged','Final 3 weeks re-scoped'], blockers=['Go-live waits for red-team evidence'], done=76, total=92)
d.risks.insert(0, RiskItem(risk='SRB requires additional red-team evidence before production', likelihood='high', impact='high', mitigation='Defer nice-to-have and close evidence checklist', owner='FDE+CISO', status='red', review_date='weekly'))
d.decisions.append(Decision(date='week19', what='Re-plan final 3 weeks', who='CUO+CISO+FDE', rationale='Protect go-live approval while preserving core benefit'))
d.demo='Go-live readiness demo: groundedness >=92%, audit trail, and red-team checklist; training analytics labeled v2.'
d.exec_message='SRB feedback changes final work plan, not the core business benefit.'
print(d.render())

In [ ]:
wrap = """# Engagement wrap — go-live handoff

Owners:
- Platform lead: Azure resources, runbooks, on-call, rollback drills.
- Product owner: adoption dashboard, roadmap, v2 prioritization.
- CUO: time-to-quote business metric and underwriter adoption.
- Underwriter SMEs: golden-set review and prompt feedback rhythm.
- CISO/DPO: SRB controls, incident escalation, retention/privacy reviews.

Links handed off:
- Week 22b SLO contract and incident runbooks.
- Week 21b SRB evidence and 25-item go-live checklist.
- Prompt/model/index release tuple and rollback procedure.
- Cost dashboard with Standard-vs-PTU revisit trigger at 200-user scale.
- Decision log, risk register, and v2 roadmap.

Hypercare: FDE remains consulted for 2 weeks, then platform/product own the operating cadence."""
print(wrap)

## Exercises
1. Rewrite the week-8 refusal-threshold memo in three paragraphs: BLUF, evidence, recommendation.
2. Add one PBI for audit completeness with functional and eval acceptance criteria.
3. Move one Option B feature to v2 and explain the business impact.
4. Add a new risk to the week-19 dashboard and assign an owner.

## Links
- Literature note: `02 Literature Notes/FDE Delivery/Customer Discovery & Stakeholder Communication — Applied`
- Snippets: `04 Code Snippets/FDE Delivery/FDE Week 23b Insurance Discovery Brief and MVP Scope Package`, `.../FDE Week 23b Steering Rhythm and Risk Conversation Dashboard`
- MOC: `06 Maps of Content/FDE Delivery Concepts`